# MiniMind-3 Self-Maintaining & Bounded Self-Modifying Loop (Google Colab)

Run cells top-to-bottom. State is atomically mirrored to Google Drive every cycle so VM disconnects resume cleanly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ed2k/minimind.git || (cd minimind && git pull)
%cd /content/minimind
!pip -q install -r requirements.txt
!python3 -m selfchange.kernel.supervisor --check-only

In [ ]:
# Step 1: Acquire MiniMind-3 + record untouched 5-Tier Baseline Scorecard
!python3 -m selfchange.bootstrap.setup_base --mode colab --output-dir selfchange/workspace/models/base
!python3 -m selfchange.bootstrap.eval_baseline --worktree . --backend minimind --model-path selfchange/workspace/models/base/minimind3_768.pth --tokenizer-path model --output selfchange/workspace/baseline_untouched.json

In [ ]:
# Steps 3-6: Progressive 3-Rung Curriculum SFT + Offline RFT/DPO + Bootstrap-Ready Gate
!python3 -m selfchange.bootstrap.warmstart_sft --work-dir selfchange/workspace/bootstrap_data --total-traces 5000 --stage full_sft
!python3 -m selfchange.bootstrap.offline_rft_dpo --rounds 1 --num-tasks 32 --num-rollouts 4
!python3 -m selfchange.bootstrap.eval_baseline --worktree . --backend scripted --output selfchange/workspace/baseline_prepared.json

In [ ]:
# Launch re-entrant autonomous loop with Google Drive mirroring
!python3 -m selfchange.colab.colab_launcher --drive-dir /content/drive/MyDrive/minimind_selfchange --max-cycles 10 --autonomy-level 1
!cat selfchange/workspace/report.md